In [1]:
import numpy as np
from pathlib import Path
import json
import pandas as pd
import pyvista as pv
import matplotlib.pyplot as plt

from phd_helpers.experiments import get_sensor_loc, parse_tekscan, F2P, build_sensor_mesh, project_sensor, project_sensor_new, project_sensor_complex
from phd_helpers.AbaqusPostprocessing import inp2pv, get_field_path, get_field_df, add_field_to_mesh, get_history_path

# Study 3
 - Updated elastico and vero material properties with currest best known values
 - Using main_inpFsteps [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 150]
 - Cartilage friction = 0.0
   - wasn't sensitive and planning to do expriments with ptfe or lubricant
 - Experiments done with ptfe sheets either side of sensor
   - Calibrated with 10x10mm vero-elastico-ptfe blocks set up

In [3]:
fea_path = Path('../../../../Computational/InpPipeline/outputs/initialFEAstuff/accuracy')
inp_file = list(fea_path.glob('study3_*/**/*.inp'))[0]
inp_file

PosixPath('../../../../Computational/InpPipeline/outputs/initialFEAstuff/accuracy/study3_35T4d5_Fsteps/inpFiles/14548R/inp/35T-neutral-00/35T-neutral-00.inp')

In [ ]:
NEED TO GET LIST OF STEPS IN CSVS DIR AND LOOP THROUGH THOSE INSTEAD OF INP_FILES 

step, frame = 0, -1
field_metrics = ["CPRESS", "U"]
history_metrics = ['CAREA', 'RF']

fea_data = {} # {F1: {'tpm':mesh1, 'mc1':mesh1}, ...} - each mesh contains all fea data
for inp_file in inp_files:
    csv_dir = inp_file.parent / 'resultCSVs' 

    # get F
    run_id = inp_file.with_suffix('').name.split('-')[-1]
    param_path = fea_path / f'study2_35T4d5/params/loop_params/{run_id}.json'
    with open(param_path, 'r') as f:
        F = json.load(f)['_loop']['max_force']

    # get meshes
    meshes = inp2pv(inp_file)
    for bone, mesh in meshes.items():
        instance = f"{bone.upper()}_INST"
        
        # Field data
        for metric in field_metrics:
            field_path = get_field_path(csv_dir, metric, step, frame, instance)
            field_df = get_field_df(field_path)
            add_field_to_mesh(mesh, field_df)

        # History data
        history_data = pd.read_csv(get_history_path(csv_dir, step))
        # F
        RF_data = history_data[history_data['historyOutputKey']=='RF1']
        RF = np.abs(RF_data['value'].iloc[frame])
        # A
        CAREA_data = history_data[history_data['historyOutputDescription']=='Total area in contact']
        CA = CAREA_data['value'].iloc[frame]

        mesh.field_data['RF'] = RF
        mesh.field_data['CA'] = CA

        #Summary data
        mesh.field_data['P_max'] = mesh['CPRESS'].max()
        mesh.field_data['P_avg'] = np.mean(mesh['CPRESS'][mesh['CPRESS']>0])
        mesh.field_data['loc_Pmax'] = np.array(mesh.points[np.argmax(mesh['CPRESS'])])

    fea_data[F] = meshes